# <center>Aggregation script<center>

#### import modules

In [ ]:
import pandas as pd
import numpy as np
import os
from functools import reduce

#### load the files

In [ ]:
# TODO write the logic of names extraction
def extract_filenames():
    # list files without windows' trash that lists opened files separately
    input_file_names = [i for i in os.listdir("input") if not "~$" in i]
    # print(input_file_names)
    tool_fn = "input/HSM_OT_master_tool_R24.xlsx"
    clean_fn = "input/UKR2206b_R24_final_anonymized_data_Dec2025.xlsx"
    raw_fn = "input/HSM_OT_Raw_Data_R24.xlsx"
    return tool_fn, clean_fn, raw_fn


In [ ]:
tool_path, clean_data_path, raw_data_path = extract_filenames()

In [ ]:
clean_data = pd.read_excel(clean_data_path)
raw_data = pd.read_excel(raw_data_path)[["_uuid", "a1_4_settlement", "settlement_label"]]
data = clean_data.merge(raw_data, left_on='uuid', right_on='_uuid', how='left')

In [ ]:
survey = pd.read_excel(tool_path, sheet_name='survey')
choices = pd.read_excel(tool_path, sheet_name='choices')

question_types = ['integer', 'select_one', 'select_multiple']
pattern = '|'.join(question_types)
survey_clean = survey[survey['type'].str.contains(pattern, na=False)].reset_index(drop=True)

In [ ]:
select_one_categorical = []
select_one_ordinal = []
select_multiple = []
numeric = []
select_multiple_options = {}
select_one_categorical_options = {}
select_one_ordinal_options_to_numbers = {}
select_one_ordinal_numbers_to_options = {}
unclear_type_cnt = 0
settlement_column = "settlement"

questions_in_order = []

for _, row in survey_clean.iterrows():
    if row["select_one_type"] == "skp":
        continue
    elif row["select_one_type"] == "set":
        settlement_column = row["name"]
        questions_in_order = [row["name"]] + questions_in_order
    elif row["type"] == "integer":
        numeric.append(row["name"])
        questions_in_order.append(row["name"])
    elif row["type"].startswith("select_multiple"):
        select_multiple.append(row["name"])
        questions_in_order.append(row["name"])

        options_list_name = row["type"].split(" ")[-1]
        options_list = choices[choices["list_name"] == options_list_name]["name"].to_list()
        select_multiple_options[row["name"]] = options_list
    elif row["type"].startswith("select_one") and row["select_one_type"] == "cat":
        select_one_categorical.append(row["name"])
        questions_in_order.append(row["name"])

        options_list_name = row["type"].split(" ")[-1]
        options_list = choices[choices["list_name"] == options_list_name]["name"].to_list()
        select_one_categorical_options[row["name"]] = options_list
    elif row["type"].startswith("select_one") and row["select_one_type"] == "ord":
        select_one_ordinal.append(row["name"])
        questions_in_order.append(row["name"])

        options_list_name = row["type"].split(" ")[-1]
        options_list = choices[choices["list_name"] == options_list_name]
        o2n = {}
        n2o = {}
        for _, choice_row in options_list.iterrows():
            o2n[choice_row["name"]] = choice_row["ord"]
            n2o[choice_row["ord"]] = choice_row["name"]
        select_one_ordinal_options_to_numbers[row["name"]] = o2n
        select_one_ordinal_numbers_to_options[row["name"]] = n2o
    else:
        unclear_type_cnt += 1
        print(row["name"] + " is of unclear type, it won't be included in the output")


#### select multiple aggregation

In [ ]:
select_multiple_columns = [sm + '/' + smo for sm in select_multiple for smo in select_multiple_options[sm]]
# this is a work-around because I did't have time to properly handle missing columns
smrm = set(['d8_1_education_barriers/no_school_in_the_settlement', 'd8_1_education_barriers/no_school_in_the_nearby_settlements', 'd8_1_education_barriers/schools_damaged', 'd8_1_education_barriers/not_enough_teachers', 'd8_1_education_barriers/not_enough_teaching_materials', 'd8_1_education_barriers/administrative_barriers', 'd8_1_education_barriers/physical_barriers', 'd8_1_education_barriers/security_barriers', 'd8_1_education_barriers/economic_barriers', 'd8_1_education_barriers/social_barriers', 'd8_1_education_barriers/other', 'd8_1_education_barriers/do_not_know', 'd8_1_education_barriers/prefer_not_to_answer'])
select_multiple_columns = [settlement_column] + [i for i in select_multiple_columns if i not in smrm]
select_multiple_data = data[select_multiple_columns]

In [ ]:
def aggregate_select_multiple(series):
    # remove NAs
    non_na = series.dropna()
    
    # if no non-NA values, return NA
    if len(non_na) == 0:
        return np.nan
    
    # if only one value, return it
    if len(non_na) == 1:
        return non_na.iloc[0]
    
    # if two values, see if there is a full consensus
    if len(non_na) == 2:
        if non_na.iloc[0] == non_na.iloc[1]:
            return non_na.iloc[0]
        else:
            return "NC"
    
    # if three or more values,
    # count occurrences of each value
    value_counts = non_na.value_counts()
    max_count = value_counts.max()
    total_count = len(non_na)
    
    # check if there's a majority (>= 50%)
    # wich will always be for just 2 values with such a rule
    if max_count / total_count >= 0.5:
        most_common_value = value_counts.idxmax()
        # a special case: if it's a 50-50 tie between 0 and 1, choose 1
        if max_count / total_count == 0.5 and total_count % 2 == 0:
            # check if we have exactly 50-50 split between 0 and 1
            if len(value_counts) == 2 and set(value_counts.index) == {0, 1}:
                return 1 
        return most_common_value
    else:
        return "NC"

select_multiple_aggregated = select_multiple_data.groupby(settlement_column).agg(aggregate_select_multiple).reset_index()

#### numeric aggregation

In [ ]:
numeric_columns = [settlement_column] + numeric
numeric_data = data[numeric_columns]

In [ ]:
def aggregate_numeric_func(series):
    # remove NAs
    data = series.dropna().tolist()
    n = len(data)
    
    # if there's just one value, aggregate to NA
    if n == 1:
        return np.nan, np.nan
    
    # if there are two values
    if n == 2:
        if data[0] == data[1]:
            return data[0], data[0]
        else:
            return "NC", "NC"
    
    # for 3 or more values
    if n >= 3:
        return np.mean(data), np.median(data)
    
    # the fallback for 0 values
    return np.nan, np.nan

def aggregate_numeric(df):
    group_col = df.columns[0]
    val_cols = df.columns[1:]
    
    results = {}
    
    for col in val_cols:
        agg_results = df.groupby(group_col)[col].apply(aggregate_numeric_func)
        
        # split the tuples into two distinct columns
        results[f"{col}_avg"] = agg_results.map(lambda x: x[0])
        results[f"{col}_median"] = agg_results.map(lambda x: x[1])
        
    return pd.DataFrame(results).reset_index()

numeric_aggregated = aggregate_numeric(numeric_data)

#### select one categorical aggregation

In [ ]:
select_one_categorical_columns = [settlement_column] + select_one_categorical
select_one_categorical_data = data[select_one_categorical_columns]

In [ ]:
def select_one_explode(df):
    # keep the first column as it is
    first_col_name = df.columns[0]
    final_df = df[[first_col_name]].copy()
    
    # iterate through the rest of the columns
    for col in df.columns[1:]:
        # prefix_sep='/' gives you the 'col_name/value' format
        # dummy_na=False ensures we don't create a column for NAs
        dummies = pd.get_dummies(df[col], prefix=col, prefix_sep='/', dummy_na=False, dtype=float)
        # stitch everything together
        final_df = pd.concat([final_df, dummies], axis=1)
        
    return final_df

In [ ]:
select_one_categorical_data_exploded = select_one_explode(select_one_categorical_data)
select_one_categorical_aggregated = select_one_categorical_data_exploded.groupby(settlement_column).agg(aggregate_select_multiple).reset_index()

#### select one ordinal aggregation

In [ ]:
select_one_ordinal_columns = [settlement_column] + select_one_ordinal
select_one_ordinal_data = data[select_one_ordinal_columns]

In [ ]:
def map_values(df, mapping_dict):
    # create a copy to avoid modifying the original dataframe
    new_df = df.copy()
    
    # get all columns except the first one
    # if you have specific columns in the mapping_dict, 
    # it's safer to iterate through the dictionary keys
    for col, mapper in mapping_dict.items():
        if col in new_df.columns:
            # .map() replaces values based on the dictionary.
            # values not in the dictionary will become NaN.
            new_df[col] = new_df[col].map(mapper)
            
    return new_df


In [ ]:
select_one_ordinal_data_mapped = map_values(select_one_ordinal_data, select_one_ordinal_options_to_numbers)

In [ ]:
def aggregate_ordinal_numeric_func(series):
    # remove NAs
    data = series.dropna().tolist()
    n = len(data)
    
    # if there's just one value, aggregate to NA
    if n == 1:
        return np.nan, np.nan
    
    # if there are two values
    if n == 2:
        if data[0] == data[1]:
            return data[0], data[0]
        else:
            return "NC", "NC"
    
    # for 3 or more values
    if n >= 3:
        return math.floor(np.mean(data) + 0.5), np.median(data)
    
    # the fallback for 0 values
    return np.nan, np.nan

def aggregate_ordinal_numeric(df):
    group_col = df.columns[0]
    val_cols = df.columns[1:]
    
    results = {}
    
    for col in val_cols:
        agg_results = df.groupby(group_col)[col].apply(aggregate_ordinal_numeric_func)
        
        # split the tuples into two distinct columns
        results[f"{col}_avg"] = agg_results.map(lambda x: x[0])
        results[f"{col}_median"] = agg_results.map(lambda x: x[1])
        
    return pd.DataFrame(results).reset_index()

In [ ]:
select_one_ordinal_data_mapped_aggregated = aggregate_ordinal_numeric(select_one_ordinal_data_mapped)

In [ ]:
def map_values_back(df, mapping_dict):
    new_df = df.copy()
    
    for col, mapper in mapping_dict.items():
        for suffix in ["_avg", "_median"]:
            target_col = col + suffix
            
            if target_col in new_df.columns:
                # use .apply() with a lambda to handle "NC" and rounding safely
                # this avoids the Reindexing error caused by .map() on mixed types
                new_df[target_col] = new_df[target_col].apply(
                    lambda x: mapper.get(int(round(float(x))), x) 
                    if isinstance(x, (int, float)) and pd.notna(x) 
                    else x
                )
                
    return new_df

In [ ]:
select_one_ordinal_aggregated = map_values_back(select_one_ordinal_data_mapped_aggregated, select_one_ordinal_numbers_to_options)

#### stitch together and save

In [ ]:
# TODO create a check for empty components

dfs = [
    select_multiple_aggregated,
    numeric_aggregated,
    select_one_categorical_aggregated,
    select_one_ordinal_aggregated
]

# we use 'reduce' to merge all dataframes one after another 
# on the first column name
first_col = dfs[0].columns[0]
df_final = reduce(lambda left, right: pd.merge(left, right, on=first_col, how='outer'), dfs)

In [ ]:
df_final.to_excel("output/datases_aggregated.xlsx", index=False)

#### ad-hocs

In [ ]:
counts = data.groupby('a1_4_settlement').size()
counts.value_counts().sort_index()


In [ ]:
# counts[counts==2]

In [ ]:
# len(set(df1_add_cols[df1_add_cols["a1_1_oblast"]=="UA74"]["a1_4_settlement"].to_list()))

### <center>Created by Andrei Andrievsky<center>